# A5 — L3 Estate Simulator

Builds a parameterised financial-institution estate and lands **real** L1/L2 findings on it. Logic lives in `src/simulator/` (CLAUDE.md rule 9); every parameter is in `src/simulator/config.py` (rule 7) and is also written out to `data/simulator/l3_simulator_parameters.csv` for provenance.

Pipeline: bank structure → assets → IAM graph → inject real findings → lifecycle (SLA, recurrence) → label.

**What is real and what is simulated.** Every finding points at a real CVE (NVD/EPSS/KEV) or a real check (Prowler/Checkov/ScoutSuite). What is simulated is *which asset a real finding landed on*, plus the estate context (criticality tier, exposure, regulatory scope, SLA, recurrence).

**The label is a formula, not ground truth** — 60% real signal + 40% simulated context, fixed in CLAUDE.md. It must never be presented the way CISA KEV is, and L3 results are reported separately from L1/L2 (rule 12). A model scoring well here is partly recovering a deterministic function of its own inputs; the informative L3 experiments are about *robustness* (label noise, weight sweeps, ablation), not raw accuracy.

> ⚠️ **Open issue before Study C** — the formula puts L1 and L2 on incommensurate scales: the L1 real component has median 0.001 against L2's 0.500, so 99.4% of vulnerability findings come out `Low` and priority is effectively "is this a misconfiguration?". See the A5 entry in CLAUDE.md's Deviations Log. Needs a supervisor decision; not changed unilaterally.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.simulator.build import build, log_stats

In [ ]:
results = build()
log_stats(results)

## The estate

In [ ]:
assets = results["assets"]
display(assets.head())
display(
    assets.groupby("business_unit").agg(
        assets=("asset_id", "count"),
        mean_tier=("criticality_tier", "mean"),
        internet_facing=("internet_facing", "mean"),
        regulated=("regulatory_scope", "mean"),
        median_blast_radius=("blast_radius", "median"),
    ).round(3)
)

## Does the label respond to its inputs?

A sanity check on the formula: every context term should move the score in the expected direction.

In [ ]:
findings = results["findings"]
for column in ["criticality_tier", "internet_facing", "regulatory_scope", "sla_breached"]:
    print(f"--- mean priority_score by {column} ---")
    print(findings.groupby(column).priority_score.mean().round(3).to_string())
    print()

## The scale problem, shown directly

This is the open issue flagged above — the two layers are not on the same scale.

In [ ]:
import pandas as pd

display(
    findings.groupby("source_layer").real_component.describe()[["mean", "50%", "max"]].round(3)
)
display(pd.crosstab(findings.source_layer, findings.priority_label, normalize="index").round(3))

## Grouped-split readiness

CLAUDE.md requires L3 to be split grouped-by-asset so the same asset never appears in both train and test. `asset_id` is carried on every finding for exactly this.

In [ ]:
print("findings:", len(findings))
print("assets   :", findings.asset_id.nunique())
print("findings per asset:", round(len(findings) / findings.asset_id.nunique(), 2))